# 17 - Decision-cell geometry: empirical evidence for the 'narrow cells' mechanism (review #11)

**Jalankan di SageMaker SETELAH `cleaned_100.pkl` + UNSW CSV tersedia.**

Mengubah hipotesis 'sel keputusan sempit' dari penjelasan spekulatif menjadi
mekanisme yang didukung empiris. Untuk model few-shot terlatih pada tiap arah,
kami ukur pada data uji target:
- **kappa** konsentrasi domain = IQR/median per fitur (ruang asli).
- **w** lebar sel efektif = jarak median tiap sampel ke ambang split XGBoost terdekat (z-space).
- **d_boundary** perturbasi minimum (L-inf, arah saliency, proyeksi PCFS) yang membalik prediksi.
- **empirical crossing probability** pada eps kecil.
- distribusi ambang split & jumlah leaf sebagai proksi granularitas.

Prediksi hipotesis: domain target CIC (varians rendah) punya w & d_boundary LEBIH KECIL daripada UNSW.
Angka NYATA -> baris LaTeX tab:decisioncell. (Menggantikan script a8_decision_cell.py yang dirujuk paper.)

In [ ]:
import importlib, subprocess, sys
for pkg,imp in [('pandas','pandas'),('numpy','numpy'),('scikit-learn','sklearn'),('xgboost','xgboost'),('boto3','boto3')]:
    try: importlib.import_module(imp)
    except ImportError: subprocess.check_call([sys.executable,'-m','pip','install','-q',pkg])
import os, json, pickle
import numpy as np, pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier

CIC_PKL='../../CICDDoS2018/data/cleaned_100.pkl'
UNSW_TRAIN='../data/UNSW_NB15_testing-set.csv'; UNSW_TEST='../data/UNSW_NB15_training-set.csv'
OUTDIR='paper2_reviewer_out'; os.makedirs(OUTDIR,exist_ok=True)
S3_BUCKET=os.environ.get('S3_BUCKET','ssh-detection-features-232032302717'); REGION='ap-southeast-1'
SEED=42; H=0.01; FEWSHOT_FRAC=0.01; NMET=4000   # subset utk metrik geometri (mahal)
EPS_GRID=np.round(np.arange(0.0,0.51,0.02),3)
print('files:', os.path.exists(CIC_PKL), os.path.exists(UNSW_TRAIN), os.path.exists(UNSW_TEST))

In [ ]:
MAP_A={'duration':('Flow Duration','dur'),'fwd_pkts':('Tot Fwd Pkts','spkts'),
       'bwd_pkts':('Tot Bwd Pkts','dpkts'),'fwd_bytes':('TotLen Fwd Pkts','sbytes'),
       'bwd_bytes':('TotLen Bwd Pkts','dbytes'),'fwd_mean':('Fwd Pkt Len Mean','smean'),
       'bwd_mean':('Bwd Pkt Len Mean','dmean'),'src_load':('Flow Byts/s','sload'),
       'dst_load':('Bwd Pkts/s','dload')}
CANON=list(MAP_A.keys()); IX={c:i for i,c in enumerate(CANON)}
def build_matrix(df,side):
    idx=0 if side=='cic' else 1; cols=[MAP_A[c][idx] for c in CANON]
    out=df[cols].copy(); out.columns=CANON; out=out.replace([np.inf,-np.inf],np.nan)
    return out.fillna(out.median(numeric_only=True)).fillna(0.0).astype(float).values
def make_xgb(seed):
    return XGBClassifier(objective='binary:logistic',eval_metric='logloss',max_depth=8,
        learning_rate=0.1,n_estimators=200,subsample=0.8,colsample_bytree=0.8,
        n_jobs=-1,random_state=seed,tree_method='hist')
def loss_bin(model,X,y):
    p=np.clip(model.predict_proba(X)[:,1],1e-15,1-1e-15); y=y.astype(float)
    return -(y*np.log(p)+(1-y)*np.log(1-p))
def saliency(model,X,y,h=H):
    n,m=X.shape; S=np.zeros((n,m))
    for i in range(m):
        Xp=X.copy(); Xp[:,i]+=h; Xm=X.copy(); Xm[:,i]-=h
        S[:,i]=(loss_bin(model,Xp,y)-loss_bin(model,Xm,y))/(2*h)
    return S
def project_functional(Xs_scaled, mean, scale, X_ref_scaled=None):
    Xo=Xs_scaled*scale+mean; Xo=np.clip(Xo,0.0,None)
    Xo[:,IX['fwd_pkts']]=np.round(Xo[:,IX['fwd_pkts']]); Xo[:,IX['bwd_pkts']]=np.round(Xo[:,IX['bwd_pkts']])
    Xo[:,IX['fwd_bytes']]=np.maximum(Xo[:,IX['fwd_bytes']],Xo[:,IX['fwd_pkts']])
    Xo[:,IX['bwd_bytes']]=np.maximum(Xo[:,IX['bwd_bytes']],Xo[:,IX['bwd_pkts']])
    if X_ref_scaled is not None:
        Xr=X_ref_scaled*scale+mean
        for j in [IX[c] for c in ['fwd_pkts','bwd_pkts','fwd_bytes','bwd_bytes','duration']]:
            Xo[:,j]=np.maximum(Xo[:,j],Xr[:,j])
    with np.errstate(divide='ignore',invalid='ignore'):
        fm=np.where(Xo[:,IX['fwd_pkts']]>0,Xo[:,IX['fwd_bytes']]/Xo[:,IX['fwd_pkts']],0.0)
        bm=np.where(Xo[:,IX['bwd_pkts']]>0,Xo[:,IX['bwd_bytes']]/Xo[:,IX['bwd_pkts']],0.0)
    Xo[:,IX['fwd_mean']]=fm; Xo[:,IX['bwd_mean']]=bm
    with np.errstate(divide='ignore',invalid='ignore'):
        dd=np.where(Xo[:,IX['duration']]>0,Xo[:,IX['duration']],np.nan)
        Xo[:,IX['src_load']]=np.nan_to_num((Xo[:,IX['fwd_bytes']]+Xo[:,IX['bwd_bytes']])/dd,nan=0.0)
        Xo[:,IX['dst_load']]=np.nan_to_num(Xo[:,IX['bwd_pkts']]/dd,nan=0.0)
    return (Xo-mean)/scale
print('util siap')

In [ ]:
with open(CIC_PKL,'rb') as f: d=pickle.load(f)
cic_feats=list(d['feature_names']); X=np.asarray(d['X'],float); sc=d.get('scaler',None)
X_orig=X*sc.scale_+sc.mean_ if (sc is not None and hasattr(sc,'scale_')) else X
cic_df=pd.DataFrame(X_orig,columns=cic_feats)
benign=d.get('label_mapping',{}).get('Benign',0); y_cic=(np.asarray(d['y'])!=benign).astype(int)
unsw_tr=pd.read_csv(UNSW_TRAIN); unsw_te=pd.read_csv(UNSW_TEST)
y_utr=unsw_tr['label'].astype(int).values; y_ute=unsw_te['label'].astype(int).values
Xc_all=build_matrix(cic_df,'cic'); Xu_tr_raw=build_matrix(unsw_tr,'unsw'); Xu_te_raw=build_matrix(unsw_te,'unsw')
print('loaded')

In [ ]:
def split_thresholds(model):
    # kumpulkan ambang split per fitur (z-space) dari dump pohon XGBoost
    df=model.get_booster().trees_to_dataframe()
    df=df[df['Feature']!='Leaf']
    thr={}
    for f in CANON:
        # nama fitur di booster: f0..f8 sesuai urutan kolom
        fi='f'+str(IX[f]); s=df[df['Feature']==fi]['Split'].dropna().values
        thr[f]=np.sort(s.astype(float))
    n_leaf=int((model.get_booster().trees_to_dataframe()['Feature']=='Leaf').sum())
    return thr,n_leaf
def eff_cell_width(X,thr):
    # w: jarak median tiap sampel ke ambang split terdekat pada fitur yg sama (z-space)
    dists=[]
    for j,f in enumerate(CANON):
        t=thr[f]
        if len(t)==0: continue
        xi=X[:,j][:,None]; dmin=np.abs(xi-t[None,:]).min(1)
        dists.append(dmin)
    D=np.stack(dists,1) if dists else np.zeros((len(X),1))
    return float(np.median(D.min(1)))   # jarak ke boundary aktif terdekat (across fitur)
def kappa_domain(Xraw):
    q1=np.percentile(Xraw,25,0); q3=np.percentile(Xraw,75,0); med=np.median(Xraw,0)
    return float(np.median((q3-q1)/(np.abs(med)+1e-9)))
def d_boundary(model,X,y,mean,scale,grid=EPS_GRID):
    # perturbasi min (L-inf, arah saliency, proyeksi PCFS) yg membalik prediksi; median atas sampel
    S=saliency(model,X,y); yp0=model.predict(X); dmin=np.full(len(X),np.nan)
    for e in grid[1:]:
        Xe=project_functional(X+e*np.sign(S),mean,scale,X_ref_scaled=X)
        flip=(model.predict(Xe)!=yp0)&(np.isnan(dmin))
        dmin[flip]=e
    return float(np.nanmedian(dmin)), float(np.mean(~np.isnan(dmin)))
def crossing_prob(model,X,y,mean,scale,e):
    S=saliency(model,X,y); yp0=model.predict(X)
    Xe=project_functional(X+e*np.sign(S),mean,scale,X_ref_scaled=X)
    return float(np.mean(model.predict(Xe)!=yp0))
print('metrik geometri siap')

In [ ]:
rng=np.random.RandomState(SEED)
Xc_tr_raw,Xc_te_raw,yc_tr,yc_te=train_test_split(Xc_all,y_cic,test_size=0.3,random_state=SEED,stratify=y_cic)
scc=StandardScaler().fit(Xc_tr_raw); scu=StandardScaler().fit(Xu_tr_raw)
Xc_tr=scc.transform(Xc_tr_raw); Xc_te=scc.transform(Xc_te_raw)
Xu_tr=scu.transform(Xu_tr_raw); Xu_te=scu.transform(Xu_te_raw)
def fewshot_model(direction,Xsrc,ysrc,Xtgt_tr,ytgt_tr):
    nfs=max(1,int(len(Xtgt_tr)*FEWSHOT_FRAC)); ifs=rng.choice(len(Xtgt_tr),nfs,replace=False)
    return make_xgb(SEED).fit(np.vstack([Xsrc,Xtgt_tr[ifs]]),np.concatenate([ysrc,ytgt_tr[ifs]]))
# CIC->UNSW: target UNSW; UNSW->CIC: target CIC
m_c2u=fewshot_model('CIC->UNSW',Xc_tr,yc_tr,Xu_tr,y_utr)
m_u2c=fewshot_model('UNSW->CIC',Xu_tr,y_utr,Xc_tr,yc_tr)
def sub(X,y,n=NMET):
    if len(X)<=n: return X,y
    i=rng.choice(len(X),n,replace=False); return X[i],y[i]
Xu_s,yu_s=sub(Xu_te,y_ute); Xc_s,yc_s=sub(Xc_te,yc_te)
rows=[]
for direction,model,Xte_raw,Xte_z,yte,mean,scale in [
    ('CIC->UNSW (target=UNSW)',m_c2u,Xu_te_raw,Xu_s,yu_s,scu.mean_,scu.scale_),
    ('UNSW->CIC (target=CIC)', m_u2c,Xc_te_raw,Xc_s,yc_s,scc.mean_,scc.scale_)]:
    thr,n_leaf=split_thresholds(model)
    w=eff_cell_width(Xte_z,thr)
    kap=kappa_domain(Xte_raw)
    dmed,frac=d_boundary(model,Xte_z,yte,mean,scale)
    cp05=crossing_prob(model,Xte_z,yte,mean,scale,0.05)
    cp10=crossing_prob(model,Xte_z,yte,mean,scale,0.10)
    rows.append({'direction':direction,'kappa':kap,'w_eff':w,'d_boundary_med':dmed,
                 'flip_frac':frac,'crossing_p_eps0.05':cp05,'crossing_p_eps0.10':cp10,'n_leaf':n_leaf})
dc=pd.DataFrame(rows); dc.to_csv(os.path.join(OUTDIR,'decision_cell.csv'),index=False)
import IPython.display as ipd; ipd.display(dc.round(4))
print('=== metrik decision-cell selesai ===')

In [ ]:
# --- baris LaTeX tab:decisioncell + cek prediksi hipotesis ---
def fmt(v,p=3):
    return '$'+f'{v:.{p}f}'.replace('.',',')+'$'
lab={'CIC->UNSW (target=UNSW)':'target UNSW (high-variance)','UNSW->CIC (target=CIC)':'target CIC (low-variance)'}
lines=[]
for _,r in dc.iterrows():
    lines.append(f"{lab[r['direction']]} & {fmt(r['kappa'])} & {fmt(r['w_eff'])} & {fmt(r['d_boundary_med'])} & {fmt(r['crossing_p_eps0.10'],3)} & {int(r['n_leaf'])} \\\\")
latex='\n'.join(lines)
open(os.path.join(OUTDIR,'decision_cell_rows.tex'),'w').write(latex)
cic=dc[dc['direction'].str.contains('CIC\\)')].iloc[0]; unsw=dc[dc['direction'].str.contains('UNSW\\)')].iloc[0]
print('Hipotesis d_boundary CIC < UNSW :', cic['d_boundary_med'] < unsw['d_boundary_med'])
print('Hipotesis w_eff CIC < UNSW      :', cic['w_eff'] < unsw['w_eff'])
print('Hipotesis kappa CIC < UNSW      :', cic['kappa'] < unsw['kappa'])
print('\n=== TEMPEL baris berikut ke Tabel tab:decisioncell ===\n'); print(latex)

In [ ]:
try:
    import boto3; s3=boto3.client('s3',region_name=REGION); up=0
    for fn in sorted(os.listdir(OUTDIR)):
        if fn.startswith('decision_cell') and fn.endswith(('.csv','.tex')):
            s3.upload_file(os.path.join(OUTDIR,fn),S3_BUCKET,f'unsw-far/paper2_reviewer/{fn}'); up+=1
    print('upload',up,'-> s3://%s/unsw-far/paper2_reviewer/'%S3_BUCKET)
except Exception as e: print('upload gagal:',e)